# NHANES 01 - Integración de ciclos y partición en desarrollo y prueba

Aplica las secciones 2 a 5 de `docs/protocolo_nhanes.md` (versión `protocolo-nhanes-v1`):

1. Descarga de los módulos de los ciclos 2007–2008 (`_E`) a 2015–2016 (`_I`) y 2017–marzo 2020 (`P_`) desde el CDC. No se descarga 2017–2018 (`_J`), cuyos participantes ya están en `P_`.
2. Integración por `SEQN` y verificación de que no haya duplicados.
3. **Partición por conglomerados** con semilla fija, usando solo ciclo, estrato y unidad primaria, **antes** de aplicar los criterios de selección y sin consultar el desenlace ni los predictores.
4. Criterios de selección y codificación de los predictores.
5. Escritura del desarrollo y de la prueba en archivos separados.

**Del conjunto de prueba solo se muestran conteos de participantes.** No se muestran su prevalencia, sus distribuciones ni sus datos faltantes: esos resultados corresponden a la evaluación final, que se ejecuta una sola vez.

In [1]:
import pandas as pd

import preparacion_nhanes as pn

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_colwidth', 80)

## 1. Descarga

Los archivos se guardan en `data/nhanes/raw/` (no se versiona). Se registran su URL, su tamaño y su SHA-256 en `particion/fuentes_nhanes.csv`, para que cualquier persona pueda comprobar que trabaja con los mismos archivos.

In [2]:
fuentes = pd.DataFrame([pn.descargar(c, m) for c in pn.CICLOS for m in pn.VARIABLES])
pn.RUTA_FUENTES.parent.mkdir(parents=True, exist_ok=True)
fuentes.to_csv(pn.RUTA_FUENTES, index=False, lineterminator='\n')
print(f"{len(fuentes)} archivos, {fuentes['bytes'].sum() / 1e6:.1f} MB")
fuentes[['ciclo', 'archivo', 'bytes', 'sha256']]

48 archivos, 136.3 MB


,ciclo,archivo,bytes,sha256
0,E,DEMO_E.xpt,3498080,02b8e97703d04bc99769c751881bece51ac9ba68cf18a2992a45d3bdfb856147
1,E,BMX_E.xpt,1800240,b67b50dbd12c9510b49b6348b595b9fd073c422c0c6eac182f90ac57b491a97d
2,E,GHB_E.xpt,111760,4c0311851825e9e0352c58af193c58d8f7a3af613aa5f522893e3c8f83e53a53
3,E,DIQ_E.xpt,2789600,fb4d698fbec26bc8a09b2cd0f3019f815114742dd2c37ccac1aaf5e0123394c5
4,E,BPQ_E.xpt,998480,b802b5855d48fc230c734083ea1e4d4a9d84e1b36bf5e78fcd725c17d6ce77df
5,E,MCQ_E.xpt,7127840,dfbe11c6b036ec4e79aeba7805833fc4951c74662a9510066c3a93932f6990e6
6,E,PAQ_E.xpt,1576000,cc4fe3815c0e3602f69db33a3709bf06b7bca38c21d6b2f74d1bb5c98a5ff35a
7,E,SMQ_E.xpt,2449520,881865ee05f4b2be0c03c761f00ad8e355f2cc5e7d3be41666edef6f3dca5862
8,F,DEMO_F.xpt,3631600,78c4d4a6e5eafa9da3bbb24a90d140fef5d8c0cda932a7ff8ee609288d4fae07
9,F,BMX_F.xpt,1890560,1fa42e07764037674c3f56e1e76f990e10c32245897597de3d731c57b91d869a


## 2. Integración y verificación de duplicados

Cada ciclo parte de `DEMO` (todos los participantes) y se le unen los demás módulos por `SEQN`. Dentro de cada módulo, `leer_ciclo` ya detiene el proceso si un `SEQN` se repite. Aquí se comprueba además que ningún `SEQN` se repita entre ciclos.

In [3]:
nhanes = pd.concat([pn.leer_ciclo(c) for c in pn.CICLOS], ignore_index=True)
duplicados = nhanes['SEQN'].duplicated().sum()
assert duplicados == 0, f'{duplicados} SEQN repetidos entre ciclos: se detiene el proceso'
print(f'Participantes integrados: {len(nhanes):,} | SEQN únicos: {nhanes.SEQN.nunique():,} | duplicados: {duplicados}')
nhanes.groupby('ciclo').agg(participantes=('SEQN', 'size'), seqn_min=('SEQN', 'min'), seqn_max=('SEQN', 'max'))

Participantes integrados: 66,148 | SEQN únicos: 66,148 | duplicados: 0


,participantes,seqn_min,seqn_max
ciclo,,,
E,10149,41475,51623
F,10537,51624,62160
G,9756,62161,71916
H,10175,73557,83731
I,9971,83732,93702
P,15560,109263,124822


Los rangos de `SEQN` de los ciclos no se superponen, lo que confirma que no hay participantes repetidos entre ciclos.

## 3. Partición por conglomerados

- Conglomerado = (ciclo, `SDMVSTRA`, `SDMVPSU`). Un conglomerado nunca se divide entre conjuntos.
- El ciclo `P_` va íntegro al desarrollo, porque se exploró antes de fijar el protocolo.
- En cada ciclo, en el orden E, F, G, H, I, los conglomerados se ordenan por (`SDMVSTRA`, `SDMVPSU`) y se permutan con `numpy.random.default_rng(20261022)`, un solo generador usado en ese orden. El primer `ceil(0.25 × n)` va a prueba.

La función `particionar` recibe solo `SEQN`, ciclo, estrato y unidad primaria.

In [4]:
particion = pn.particionar(nhanes[['SEQN', 'ciclo', 'SDMVSTRA', 'SDMVPSU']])
particion.to_csv(pn.RUTA_PARTICION, index=False, lineterminator='\n')

# Comprobaciones
assert particion.groupby('conglomerado')['conjunto'].nunique().max() == 1, 'un conglomerado quedó dividido'
assert (particion.loc[particion.ciclo == 'P', 'conjunto'] == 'desarrollo').all()
assert particion['SEQN'].is_unique

print('Archivo:', pn.RUTA_PARTICION.relative_to(pn.RAIZ).as_posix())
print('SHA-256:', pn.sha256(pn.RUTA_PARTICION))

Archivo: particion/particion_nhanes.csv
SHA-256: 2f05d104399e7e063d9f142d0532ccced9b500ba7cf5f93611f56980fd087fcd


In [5]:
resumen = (particion.groupby(['ciclo', 'conjunto'])
           .agg(conglomerados=('conglomerado', 'nunique'), participantes=('SEQN', 'size'))
           .unstack('conjunto', fill_value=0))
resumen.loc['Total'] = resumen.sum()
resumen

conglomerados        participantes       
conjunto    desarrollo prueba    desarrollo prueba
ciclo                                             
E                   24      8          7534   2615
F                   23      8          7817   2720
G                   23      8          7388   2368
H                   22      8          7472   2703
I                   22      8          7251   2720
P                   49      0         15560      0
Total              163     40         53022  13126

In [6]:
total_cong = particion['conglomerado'].nunique()
cong_prueba = particion.loc[particion.conjunto == 'prueba', 'conglomerado'].nunique()
print(f'Conglomerados en prueba: {cong_prueba} de {total_cong} ({100 * cong_prueba / total_cong:.1f}%)')
print(f"Participantes de DEMO en prueba: {100 * (particion.conjunto == 'prueba').mean():.1f}%")

Conglomerados en prueba: 40 de 203 (19.7%)
Participantes de DEMO en prueba: 19.8%


## 4. Criterios de selección (sección 3)

Se aplican por separado a cada conjunto, en el orden del protocolo. `DIQ160`, `DIQ050` y `DIQ070` faltantes (por salto del cuestionario) no excluyen. Del conjunto de prueba solo se muestra cuántos participantes quedan tras cada criterio, un dato que la tesis necesita para describir la muestra.

In [7]:
nhanes = nhanes.merge(particion[['SEQN', 'conjunto']], on='SEQN', validate='one_to_one')
seleccion, flujos = {}, {}
for conjunto in ['desarrollo', 'prueba']:
    seleccion[conjunto], conteo = pn.flujo_seleccion(nhanes[nhanes.conjunto == conjunto])
    flujos[conjunto] = dict(conteo)
flujo = pd.DataFrame(flujos)
flujo['total'] = flujo.sum(axis=1)
flujo

,desarrollo,prueba,total
Participantes en DEMO,53022,13126,66148
Edad 20 a 69 años,25576,6178,31754
Sin embarazo (RIDEXPRG distinto de 1),25243,6107,31350
DIQ010 = 2 (sin diagnóstico de diabetes ni borderline),21811,5329,27140
Sin aviso de prediabetes (DIQ160 distinto de 1),20202,5045,25247
Sin insulina ni pastillas (DIQ050 y DIQ070 distintos de 1),20198,5042,25240
HbA1c disponible,18264,4627,22891


## 5. Codificación y escritura de los conjuntos

Los registros con predictores faltantes **se conservan**: la imputación se hace dentro del `Pipeline` en el entrenamiento (sección 6). El desenlace es `hba1c_elevada` (HbA1c ≥ 5.7 %); el secundario es `hba1c_diabetes` (≥ 6.5 %). El peso combinado (`peso`) solo se usa para descriptivos.

In [8]:
desarrollo = pn.codificar(seleccion['desarrollo'])
prueba = pn.codificar(seleccion['prueba'])
assert set(desarrollo.SEQN).isdisjoint(prueba.SEQN)

pn.RUTA_DESARROLLO.parent.mkdir(parents=True, exist_ok=True)
desarrollo.to_csv(pn.RUTA_DESARROLLO, index=False, lineterminator='\n')
prueba.to_csv(pn.RUTA_PRUEBA, index=False, lineterminator='\n')

print(f'Desarrollo: {len(desarrollo):,} participantes, {desarrollo.conglomerado.nunique()} conglomerados -> {pn.RUTA_DESARROLLO.name}')
print(f'Prueba:     {len(prueba):,} participantes, {prueba.conglomerado.nunique()} conglomerados -> {pn.RUTA_PRUEBA.name} (reservado)')
print('SHA-256 prueba:', pn.sha256(pn.RUTA_PRUEBA))
del prueba, seleccion  # el resto del notebook solo usa el desarrollo

Desarrollo: 18,264 participantes, 163 conglomerados -> nhanes_desarrollo.csv
Prueba:     4,627 participantes, 40 conglomerados -> nhanes_prueba.csv (reservado)
SHA-256 prueba: c48492938873def3d096e59c8d465e32843fdd574d3dd2b349e90754d58ee62c


### Descripción del conjunto de desarrollo

Prevalencia del desenlace (sin ponderar y ponderada con el peso combinado) y datos faltantes por predictor.

In [9]:
def ponderada(x, w):
    m = x.notna() & w.notna() & (w > 0)
    return (x[m] * w[m]).sum() / w[m].sum()

prev = pd.DataFrame({
    'n': desarrollo.groupby('ciclo').size(),
    'HbA1c >= 5.7 (%)': 100 * desarrollo.groupby('ciclo')[pn.DESENLACE].mean(),
    'HbA1c >= 6.5 (%)': 100 * desarrollo.groupby('ciclo')[pn.DESENLACE_SECUNDARIO].mean(),
})
prev.loc['Total'] = [len(desarrollo), 100 * desarrollo[pn.DESENLACE].mean(), 100 * desarrollo[pn.DESENLACE_SECUNDARIO].mean()]
prev['n'] = prev['n'].astype(int)
print(f"Ponderada (peso combinado): HbA1c >= 5.7 {100 * ponderada(desarrollo[pn.DESENLACE], desarrollo.peso):.1f}% | "
      f">= 6.5 {100 * ponderada(desarrollo[pn.DESENLACE_SECUNDARIO], desarrollo.peso):.1f}%")
prev

Ponderada (peso combinado): HbA1c >= 5.7 20.6% | >= 6.5 1.5%


,n,HbA1c >= 5.7 (%),HbA1c >= 6.5 (%)
ciclo,,,
E,2668,24.8501,2.8111
F,2941,29.5478,2.4821
G,2648,27.4169,2.8701
H,2731,23.9839,1.6111
I,2405,26.0291,2.1206
P,4871,27.9203,2.2993
Total,18264,26.8233,2.3598


In [10]:
faltantes = pd.DataFrame({
    'faltantes': desarrollo[pn.PREDICTORES].isna().sum(),
    '%': 100 * desarrollo[pn.PREDICTORES].isna().mean(),
})
print(f'Registros con todos los predictores: {desarrollo[pn.PREDICTORES].notna().all(axis=1).sum():,} de {len(desarrollo):,}')
faltantes

Registros con todos los predictores: 17,324 de 18,264


,faltantes,%
edad,0,0.0000
hombre,0,0.0000
imc,149,0.8158
cintura,623,3.4111
hta,13,0.0712
familiar_dm,283,1.5495
actividad,0,0.0000
fuma,11,0.0602


In [11]:
desarrollo[['edad', 'imc', 'cintura', 'hba1c']].describe().T

,count,mean,std,min,25%,50%,75%,max
edad,18264.0000,42.5208,14.0179,20.0000,30.0000,42.0000,54.0000,69.0000
imc,18115.0000,28.7826,6.9282,13.6395,23.9672,27.6487,32.2157,92.2848
cintura,17641.0000,97.2626,16.2497,55.5000,85.6000,95.7000,106.9000,178.2000
hba1c,18264.0000,5.4621,0.5868,3.6000,5.2000,5.4000,5.7000,14.5000


## Registro

- `particion/particion_nhanes.csv` y `particion/fuentes_nhanes.csv` se suben a Git **antes** de cualquier modelado. El SHA-256 de la partición se anota en la sección 15 del protocolo.
- `data/processed/nhanes_prueba.csv` no se abre en ningún notebook hasta la evaluación final.